#### MODEL TRAINING.

#### 1.1) Import Data and Required Packages.

Importing pandas,Numpy,seaborn,matplotlib,and Warnings Library.

In [ ]:
!pip install catboost

In [ ]:
## Basic Imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Modelling
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

#### Import the CSV data as Pandas Dataframe.

In [38]:
df = pd.read_csv("data/study.csv", sep="\t")

df.head(6)

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
5,female,group B,associate's degree,standard,none,71,83,78


In [39]:
df.shape,df.columns

((1000, 8),
 Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
        'test preparation course', 'math score', 'reading score',
        'writing score'],
       dtype='str'))

#### Create a dataframe.

In [40]:
df=pd.DataFrame(df)

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


#### Creating new features of 'total_score' and 'Average'.

In [41]:
df['total_score']=df['math score']+df['reading score']+df['writing score']
df['Average']=df['total_score']/3


df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,total_score,Average
0,female,group B,bachelor's degree,standard,none,72,72,74,218,72.666667
1,female,group C,some college,standard,completed,69,90,88,247,82.333333
2,female,group B,master's degree,standard,none,90,95,93,278,92.666667
3,male,group A,associate's degree,free/reduced,none,47,57,44,148,49.333333
4,male,group C,some college,standard,none,76,78,75,229,76.333333


#### Preaparing X and Y variable [X->Independent/Input] and [Y->Dependent/Output]

In [42]:
## Creating an Input 'X'.

X=df.drop(columns=['math score'])

X.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score,total_score,Average
0,female,group B,bachelor's degree,standard,none,72,74,218,72.666667
1,female,group C,some college,standard,completed,90,88,247,82.333333
2,female,group B,master's degree,standard,none,95,93,278,92.666667
3,male,group A,associate's degree,free/reduced,none,57,44,148,49.333333
4,male,group C,some college,standard,none,78,75,229,76.333333


In [43]:
print("Categories in 'gender' variable :  ",end=" ")
print(df['gender'].unique())

print("Categories in 'race/ethnicity' variable :  ",end=" ")
print(df['race/ethnicity'].unique())

print("Categories in 'parental level of education' variable :  ",end=" ")
print(df['parental level of education'].unique())

print("Categories in 'lunch' variable :  ",end=" ")
print(df['lunch'].unique())

print("Categories in 'test preparation course' variable :  ",end=" ")
print(df['test preparation course'].unique())

Categories in 'gender' variable :   <ArrowStringArray>
['female', 'male']
Length: 2, dtype: str
Categories in 'race/ethnicity' variable :   <ArrowStringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str
Categories in 'parental level of education' variable :   <ArrowStringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str
Categories in 'lunch' variable :   <ArrowStringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
Categories in 'test preparation course' variable :   <ArrowStringArray>
['none', 'completed']
Length: 2, dtype: str


In [44]:
## Creating an Output 'Y'.

Y=df['math score']

Y.head()

0    72
1    69
2    90
3    47
4    76
Name: math score, dtype: int64

#### Create column transformer with 3 types of Transformers.

Mainly OneHotEncoder use for only 'categorical_features'
And StandardScaler use for only 'numerical_features'.

In [45]:
num_features=X.select_dtypes(exclude="str").columns
cat_features=X.select_dtypes(include="str").columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
# 'ColumnTransformer' is responsible for transforming my columns/features 
# by 'OneHotEncoder' for 'categorical_features' and 'StandardScaler' for 'numerical_feature'.


oh_transformer=OneHotEncoder()
numeric_transformer=StandardScaler()

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",oh_transformer,cat_features),
        ("StandardScaler",numeric_transformer,num_features),
    ]
)

In [46]:
X=preprocessor.fit_transform(X)



In [54]:
X ,X.shape

(array([[ 1.        ,  0.        ,  0.        , ...,  0.39149181,
          0.34357423,  0.34357423],
        [ 1.        ,  0.        ,  0.        , ...,  1.31326868,
          1.0219275 ,  1.0219275 ],
        [ 1.        ,  0.        ,  0.        , ...,  1.64247471,
          1.74706375,  1.74706375],
        ...,
        [ 1.        ,  0.        ,  0.        , ..., -0.20107904,
         -0.19443008, -0.19443008],
        [ 1.        ,  0.        ,  0.        , ...,  0.58901542,
          0.46053169,  0.46053169],
        [ 1.        ,  0.        ,  0.        , ...,  1.18158627,
          1.06871048,  1.06871048]], shape=(1000, 21)),
 (1000, 21))

In [52]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.193999,0.391492,0.343574,0.343574
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.427476,1.313269,1.021927,1.021927
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.770109,1.642475,1.747064,1.747064
3,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.833899,-1.583744,-1.293830,-1.293830
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.605158,0.457333,0.600881,0.600881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,2.044215,1.774157,1.840630,1.840630
996,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.970952,-0.859491,-0.732434,-0.732434
997,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.125472,-0.201079,-0.194430,-0.194430
998,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.605158,0.589015,0.460532,0.460532


#### Saperate Dataset into Train and Test Split.

In [59]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.23,random_state=40)

X_train.shape,X_test.shape,Y_train.shape,Y_test.shape

((770, 21), (230, 21), (770,), (230,))

#### Create an Evaluate Function to give all metrics after Model Training.